In [1]:
from pyspark.sql import SparkSession
from pyspark.mllib.tree import RandomForest
from pyspark.mllib.regression import LabeledPoint
from pyspark.mllib.evaluation import MulticlassMetrics
from pyspark.mllib.linalg import Vectors
from pyspark.sql.functions import when
import numpy as np


In [2]:
spark = SparkSession.builder.appName("RandomForest").getOrCreate()
data = spark.read.csv("data.csv", header = True, inferSchema=True)

In [3]:
# M is encoded as 1.0 and B is encoded as 0.0
data = data.withColumn("label", 
                              when(data.diagnosis == "M", 1.0)
                              .otherwise(0.0))


feature_cols = [col for col in data.columns if col not in ['id', 'diagnosis', 'label']]
model_data = data.select(feature_cols + ['label'])

def create_labeled_point(row):
    features = [row[col] for col in feature_cols]
    return LabeledPoint(row.label, Vectors.dense(features))

rdd_data = model_data.rdd.map(create_labeled_point)

In [4]:
train_rdd, test_rdd = rdd_data.randomSplit([0.7, 0.3])

In [ ]:
rf = RandomForest.trainClassifier(data=train_rdd,
                                  numClasses=2,
                                  categoricalFeaturesInfo={},
                                  numTrees=10,
                                  featureSubsetStrategy="auto",
                                  impurity='gini',
                                  maxDepth=3,
                                  maxBins=32)
predictions = rf.predict(test_rdd.map(lambda x: x.features))
predictions_and_labels = test_rdd.map(lambda x: x.label).zip(predictions)

[(1.0, 1.0), (1.0, 1.0), (1.0, 1.0), (1.0, 1.0)]

In [8]:
metrics = MulticlassMetrics(predictions_and_labels)
accuracy = metrics.accuracy
precision_1 = metrics.precision(1.0)
recall_1 = metrics.recall(1.0)
f1_1 = metrics.fMeasure(1.0)

c:\Users\Jason\Documents\GitHub\CSCI-49376\myenv\lib\site-packages\pyspark\sql\context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [10]:
print(f"\n=== MODEL EVALUATION METRICS ===")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision_1:.4f}")
print(f"Recall: {recall_1:.4f}")
print(f"F1-Score: {f1_1:.4f}")


=== MODEL EVALUATION METRICS ===
Accuracy: 0.9595
Precision: 0.9545
Recall: 0.9403
F1-Score: 0.9474


In [14]:
confusion_matrix = metrics.confusionMatrix()
print(f"\n=== CONFUSION MATRIX ===")
print(confusion_matrix.toArray())


=== CONFUSION MATRIX ===
[[103.   3.]
 [  4.  63.]]
